# AI-Powered Medical Image Analysis System
## Training Pipeline

This notebook covers the step-by-step implementation of evaluating chest X-ray images to detect Pneumonia.

### Step 1: Load and Preview the Dataset
Ensure you have downloaded the Chest X-Ray dataset from Kaggle and placed it in the `data/chest_xray/` folder.

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# Sample paths - update if needed
train_dir = '../data/chest_xray/train'
pneumonia_dir = os.path.join(train_dir, 'PNEUMONIA')

if os.path.exists(pneumonia_dir):
    sample_img_name = os.listdir(pneumonia_dir)[0]  # Get first image
    image_path = os.path.join(pneumonia_dir, sample_img_name)
    
    # Load image
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    
    # Show image
    plt.imshow(image, cmap='gray')
    plt.title("Sample Pneumonia X-Ray Image")
    plt.show()
else:
    print("Dataset not found. Please extract Kaggle Data to data/chest_xray/")

### Step 2: Data Preprocessing & Augmentation
We use TensorFlow's `ImageDataGenerator` to normalize and augment our data to improve robustness.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define Image Data Generator for Augmentation
datagen = ImageDataGenerator(
    rescale=1./255, 
    rotation_range=10,
    width_shift_range=0.1, 
    height_shift_range=0.1, 
    shear_range=0.1, 
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# Load training dataset (using subset for split if val doesn't exist natively)
print("Loading Training Data...")
train_data = datagen.flow_from_directory(
    train_dir, 
    target_size=(256, 256), 
    color_mode="grayscale", 
    batch_size=32, 
    class_mode="binary"
)

### Step 3: Model Building
We construct a custom Convolutional Neural Network (CNN) specifically tailored for analyzing medical spatial geometry.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Build CNN model
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(256, 256, 1)),
    MaxPooling2D(pool_size=(2,2)),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.summary()

### Step 4: Training
Compile and train the model.

In [ ]:
# Compile Model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train Model
history = model.fit(train_data, epochs=10)

# Save Model for deployment
import os
os.makedirs('../models', exist_ok=True)
model.save('../models/medical_ai_model.h5')
print("Model Successfully Saved!")

### Step 5: Evaluation & Visualization
Let's see how our model performs.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score
import seaborn as sns
import numpy as np

test_dir = '../data/chest_xray/test'

test_datagen = ImageDataGenerator(rescale=1./255)
test_data = test_datagen.flow_from_directory(
    test_dir, target_size=(256, 256), color_mode="grayscale", 
    batch_size=32, class_mode="binary", shuffle=False
)

# Predict on test images
y_true = test_data.classes
y_pred_probs = model.predict(test_data)
y_pred = np.where(y_pred_probs > 0.5, 1, 0)

# Compute accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Model Accuracy: {acc * 100:.2f}%")

# Generate Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=test_data.class_indices.keys(), yticklabels=test_data.class_indices.keys())
plt.xlabel('Predicted Diagnosis')
plt.ylabel('True Diagnosis')
plt.title('Confusion Matrix')
plt.show()